# RCLAgent: 4 trace train trên Kaggle GPU

Notebook smoke test với backbone thay thế. Giữ nguyên code/prompt/tool tác giả tại
`3052e7a6a9598d984ba1482f3dc41123c8823a99`; giữ suy luận theo span và gộp đệ quy.
Chưa tái hiện Table I, chưa đo độ chính xác, chưa chạy H1/IM1/R-a.

**Trước khi chạy:** để notebook **Private**, tạo một dataset **Private** chỉ chứa
`rclagent-kaggle-bundle.zip`, rồi Add Input dataset đó. Bật **Internet** và GPU.
Cấu hình cố định này hỗ trợ GPU compute capability ≥7.5, ít nhất 13.000 MiB VRAM trống
trên **một** thiết bị; không hỗ trợ P100. Nếu được T4 ×2, chỉ dùng một T4.

Chạy các cell từ trên xuống. Cell C–F chạy mỗi trace **đúng một lần**; lỗi sẽ dừng và xuất log.
Nếu phiên bị ngắt, chạy lại cell C–F, hoặc gắn ZIP kết quả cũ làm input của phiên mới,
để chạy tiếp các trace còn lại. Trace đã xong, kể cả thất bại, không chạy lại.
Không tự xóa run-once.json hay progress.json để thử lại, đổi model hoặc tăng trần.


## Kết quả Kaggle thực tế đã ghi nhận

Version 3 đã nạp model trên **một Tesla T4**, qua A/B/C nhưng dừng ở D:
request tổng hợp đầu tiên nhận HTTP 500 do Triton attention kernel cần 98.304 byte
shared memory, phần cứng giới hạn 65.536 byte. Có **1 request tổng hợp, 0 request
trên trace benchmark**. Chưa xác minh tool_calls/round trip/print_results; chưa chạy E
hoặc scorer. Process đã được dọn và log đã xuất.

Theo yêu cầu sửa lỗi tiếp theo, bản này bổ sung patch tương thích vLLM có kiểm hash:
tile truy vấn 64 cho Turing, giữ 128 trên kiến trúc khác. Không sửa RCLAgent.
Cell B tái hiện lỗi kernel gốc rồi kiểm bản vá với dense attention FP32 trên GPU,
15 trường hợp gồm ranh giới tile và cached context. Chỉ qua kiểm này mới nạp model.
Kết quả version 3 là lịch sử thất bại. Kết quả bản sửa được ghi riêng khi có output native.


## Kết quả bản sửa: Kaggle version 5

Đã kiểm thật trên một T4: kernel gốc tái hiện lỗi 96/64 KiB, bản vá qua 15 kiểm
dense attention FP32; lỗi cache Python của version 4 đã sửa. A/B/C và **D đạt**:
3 request tổng hợp HTTP 200, structured tool_calls, role=tool, print_results và
kiểm trường request đều qua.

**E dừng sau 2 model requests, khoảng 24,94 giây.** Khi đang kiểm root span
frontendservice, model yêu cầu search_logs và search_fluctuating_metrics cho
cartservice tại timestamp 1708475368662. Cartservice không thuộc trace/domain
đóng gói; guard chặn trước thực thi query. Chưa có kết quả cuối hoặc điểm chấm.
Server/process đã dọn; log và phần đã hoàn thành được giữ trong ZIP.

Đây là kết quả smoke bị dừng ở E, không phải pass end-to-end hoặc kết luận
về accuracy/phương pháp gốc. Không mở rộng dữ liệu, sửa prompt/tool hay chạy lại
để chọn đáp án trong lượt đó.


## Version 6: phạm vi query theo cửa sổ thời gian

Chẩn đoán sau version 5: tool tác giả nhận mọi service_name/timestamp và phục vụ
toàn bộ dữ liệu gộp; dữ liệu đầy đủ có log cartservice trong ±60 giây quanh root span.
Query đó hợp lệ theo contract tác giả; gói cũ chỉ chứa các cặp service/giây của span
nên guard chặt hơn tác giả. Bản này đóng gói mọi dòng metric trong ±1.200 giây và mọi
dòng log trong ±60 giây quanh các giây của trace, mọi service, đều thuộc đúng ca này;
phản hồi endpoint gốc khớp byte với dữ liệu đầy đủ (cpu-equivalence.json). Guard cho
mọi service, chỉ chặn timestamp ngoài cửa sổ. Không đổi prompt, tool, model hoặc trần.

Rủi ro đã biết trước khi chạy: kết quả log cartservice gốc khoảng 125 nghìn token.
Nếu model gọi lại query này, request tiếp theo vượt context và lượt chạy dừng vì
overflow, không cắt ngữ cảnh.


## Kết quả version 6

D đạt. E dừng sau 2 model requests: model hỏi cartservice, tool trả đúng dữ liệu đầy đủ
(272.734 byte), và request thứ ba cần 126.088 token, vượt context 16.384. Guard dừng
trước khi gửi, không cắt ngữ cảnh. Chưa có kết quả cuối hoặc điểm chấm; process đã dọn.
Quét CPU 450 error trace của 90 ca RE2-OB: mọi trace đều có query search_logs hợp lệ
(cartservice) vượt context ngay vòng tool đầu.


## Từ version 7: ngân sách kết quả search_logs (sai lệch so với tác giả)

Theo quyết định của chủ dự án, wrapper ngoài upstream đặt ngân sách **2.048 token**
(tokenizer Qwen đúng revision) cho mỗi kết quả `search_logs`, áp như nhau cho mọi ca.
Kết quả trong ngân sách được trả **nguyên byte**. Kết quả lớn hơn được rút gọn theo bậc,
chọn bước đầu tiên vừa ngân sách: gộp dòng trùng hệt (giữ số lần, giây đầu/cuối),
rồi gộp theo mẫu thông điệp (che UUID và từ có chữ số thành `<*>`, giữ số lần và giá trị
nguyên văn đầu tiên), cuối cùng mới cắt theo ưu tiên severity, có ghi rõ `truncated`.
Model thấy một dòng thông báo `[harness tool-result budget ...]` ở đầu kết quả.
Từ bản batch, các `search_logs` trong cùng một lượt trả lời của model chia đều ngân sách:
mỗi kết quả nhận ⌊2.048 / n⌋ token. Khi lượt chỉ có một `search_logs` (mọi lượt của
version 7 và 8), ngân sách vẫn là 2.048 và kết quả không đổi byte nào.
Kết quả metric, prompt, model, sampling, context, output, retry, số vòng tool và các trần
giữ nguyên; guard vẫn dừng khi vượt context, không cắt ngữ cảnh.

Kiểm CPU trước khi chạy: query đúng chỉ dẫn của prompt không đổi byte nào trên 450 trace;
không bỏ dòng log hay dòng ERROR nào; tái dựng request thứ ba của version 6 từ 126.088
còn 1.589 token. Kết quả của notebook này là của bản thích nghi có ngân sách,
không so trực tiếp với Table I.


## Kết quả version 7

Ngân sách hoạt động: root span lại hỏi cartservice, phản hồi 124.953 token được gộp
còn 454 token, không bỏ dòng nào. E chạy 60 requests HTTP 200 trong 620,7 giây,
27/33 span qua self-state, prompt lớn nhất 1.719 token. Lượt chạy dừng ở request 60:
model gọi search_fluctuating_metrics với timestamp `1808475368747` (sai một chữ số),
guard chặn vì ngoài cửa sổ trace. Chưa có kết quả cuối hoặc điểm chấm; process đã dọn.


## Từ version 8: timestamp ngoài toàn bộ dữ liệu

Trên dữ liệu đầy đủ, tool tác giả trả rỗng cho query của version 7 vì mọi dòng metric/log
nằm trong giây [1.705.339.822, 1.708.489.861]; gói cũng trả rỗng, nên guard version 7
chặt hơn tác giả. Guard nay cho qua thêm query có cửa sổ endpoint (±1.200 giây metric,
±60 giây log) nằm hoàn toàn ngoài khoảng dữ liệu đầy đủ, ghi trong `selection.json`.
Mọi timestamp khác ngoài cửa sổ trace vẫn bị chặn. Không đổi prompt, tool, model,
ngân sách hay các trần; cùng trace và cấu hình với version 7.


## Kết quả version 8

Chạy trọn A–F trên một T4: 106 model requests trong 1.332,9 giây, 38 kết quả search_logs
(1 kết quả được rút gọn), có kết quả cuối và điểm chấm của tác giả cho trace này.
Đây là một trace smoke, không phải ước lượng độ chính xác.


## Bản này: chạy nhiều trace train

Gói chứa một thư mục con cho mỗi trace (`traces/<trace_id>/`): dữ liệu của riêng trace,
`selection.json`, báo cáo tương đương CPU và nhãn chấm điểm. Mỗi gói con được đối chiếu
endpoint gốc với dữ liệu đầy đủ như version 6–8. Các trace của lượt này:

- `ad1c27da086fee80da27cbdf90e7b6e8` (currencyservice_disk_3, 25 span)
- `7bbfa6060d6e01eed1ac0ec2ceead992` (recommendationservice_socket_1, 23 span)
- `272a47c21e51eefb241d43403e3b2c26` (recommendationservice_cpu_2, 25 span)
- `29583dc604a5ce5fdd4daab6bf3144c4` (emailservice_delay_1, 31 span)

Gói có 36 lượt chạy: mỗi trace chạy một lần cho mỗi điều kiện (no_metric, metric, metric_no_log) và mỗi seed (20261026, 20261027, 20261028). Mỗi lượt là một đơn vị chạy đúng một lần, khóa theo `run_id`.

**Cấu hình kiểm độ nhạy về trần:** 8 vòng tool mỗi span, 320 model request tính cả retry, deadline 2400 giây mỗi lượt (cấu hình chính: 4, 160, 1800). Model, prompt, tool, sampling, context, output và retry giữ nguyên. Kết quả báo cáo tách khỏi cấu hình chính.

Thứ tự chạy theo gói; mỗi trace có tool server, trần và deadline riêng. Không đổi prompt,
tool, model, sampling, context, output, retry, số vòng tool hay các trần.


## Từ version 11: server không chunked prefill

Đến version 10, server prefill theo khối 2.048 token. Trên T4, prompt tổng hợp 8.446 token
của trace 45 span cần khoảng 232 giây, vượt timeout 180 giây của upstream. Từ version 11,
server prefill cả prompt trong một bước (`--max-num-batched-tokens` bằng context, không
`--enable-chunked-prefill`). Lượt đo riêng cho prompt đó còn 9,4 giây, và câu trả lời của
5 prompt đo được giống hệt cấu hình cũ. Cell C dừng nếu log vLLM cho thấy chunked prefill
vẫn bật. Đây là thay đổi cấu hình server, không đổi các thiết lập đóng băng.


In [ ]:
# Bootstrap chuẩn thư viện Python; chưa cài hay chạy model.
import hashlib, importlib.util, json
from pathlib import Path
EXPECTED_BUNDLE_SHA256 = '149997c67f0ce3014d885e1397300ca41504f2b0d58c8f83398b2f7f3dcc9cef'
EXPECTED_MANIFEST_SHA256 = '0b43d32a32071999bf208573298c4d1a4bf12a24b22bfc979add27eb6c39ed2b'
RUNTIME_SOURCE = '"""Kaggle bootstrap, localhost model lifecycle, one protocol gate and one trace.\n\nThis file uses only the standard library until dependencies have been installed.\n"""\nimport atexit\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport platform\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nimport traceback\nimport venv\nimport zipfile\nfrom pathlib import Path\n\nMODEL = \'Qwen/Qwen3-4B-Instruct-2507\'\nREVISION = \'cdbee75f17c01a7cc42f958dc650907174af0554\'\nVLLM_VERSION = \'0.10.2\'\nCOMMIT = \'3052e7a6a9598d984ba1482f3dc41123c8823a99\'\nPINS = [\'vllm==0.10.2\', \'pandas==2.2.3\', \'numpy==2.2.6\', \'flask==3.1.2\',\n        \'requests==2.32.5\', \'tenacity==9.1.2\', \'jsonschema==4.25.1\', \'psutil==7.0.0\',\n        \'transformers==4.55.4\', \'huggingface-hub==0.34.4\']\nPINS += [\'openai==1.101.0\', \'fastapi==0.116.1\', \'pydantic==2.11.7\', \'tokenizers==0.21.4\']\nROOT = Path(\'/kaggle/working/rclagent-smoke\')\nINPUT = Path(\'/kaggle/input\')\nTEMP = Path(\'/tmp/rclagent-smoke\')\nPYTHON = TEMP / \'env/bin/python\'\n# Per-run trace limits; a manifest\'s harness_limits overrides them for a sensitivity bundle.\nHARNESS_LIMITS = {\'max_model_requests_including_retry\': 160, \'rcl_deadline_seconds\': 1800,\n                  \'max_tool_rounds_per_span\': 4}\nLIMIT_FLAGS = {\'max_model_requests_including_retry\': \'--max-requests\', \'rcl_deadline_seconds\': \'--deadline-seconds\',\n               \'max_tool_rounds_per_span\': \'--tool-rounds\'}\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(4 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef dump(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef kaggle_only():\n    if sys.platform != \'linux\' or not INPUT.is_dir():\n        raise RuntimeError(\'GPU/model execution is permitted only inside Kaggle, never Windows\')\n\n\ndef environment():\n    kaggle_only()\n    ROOT.mkdir(parents=True, exist_ok=True)\n    result = {\'python\': sys.version, \'platform\': platform.platform(),\n              \'disk\': {str(p): shutil.disk_usage(p)._asdict() for p in (Path(\'/tmp\'), Path(\'/kaggle/working\'))}}\n    with open(\'/proc/meminfo\', encoding=\'utf-8\') as f:\n        result[\'ram_kib\'] = {line.split(\':\')[0]: int(line.split()[1]) for line in f if line.startswith((\'MemTotal:\', \'MemAvailable:\'))}\n    query = subprocess.run([\'nvidia-smi\', \'--query-gpu=index,name,uuid,memory.total,memory.free,driver_version,compute_cap\',\n                            \'--format=csv,noheader,nounits\'], capture_output=True, text=True)\n    result[\'nvidia_smi_exit\'] = query.returncode\n    result[\'gpus\'] = []\n    for line in query.stdout.splitlines():\n        fields = [x.strip() for x in line.split(\',\')]\n        if len(fields) != 7:\n            continue\n        index, name, uuid, total, free, driver, capability = fields\n        result[\'gpus\'].append({\'index\': int(index), \'name\': name, \'uuid\': uuid,\n                              \'total_mib\': float(total), \'free_mib\': float(free),\n                              \'driver\': driver, \'compute_capability\': float(capability)})\n    probe = subprocess.run([sys.executable, \'-c\', \'import json,torch; print(json.dumps({"torch":torch.__version__,"cuda":torch.version.cuda,"cuda_available":torch.cuda.is_available()}))\'], capture_output=True, text=True)\n    result[\'base_torch_probe\'] = {\'exit\': probe.returncode, \'stdout\': probe.stdout, \'stderr\': probe.stderr}\n    dump(ROOT / \'environment-before.json\', result)\n    print(json.dumps(result, indent=2))\n    if query.returncode or not result[\'gpus\']:\n        raise RuntimeError(\'No GPU allocated; enable a Kaggle GPU before continuing\')\n    # One device only. No summed VRAM and no tensor parallelism across devices.\n    eligible = [g for g in result[\'gpus\'] if g[\'compute_capability\'] >= 7.5 and g[\'free_mib\'] >= 13000]\n    if not eligible:\n        raise RuntimeError(\'This fixed vLLM/FP16 configuration needs one GPU with compute capability >=7.5 and >=13000 MiB free; unsupported allocation stops here\')\n    if result[\'ram_kib\'][\'MemAvailable\'] < 10 * 1024**2:\n        raise RuntimeError(\'Less than 10 GiB host RAM available\')\n    if shutil.disk_usage(\'/tmp\').free < 22 * 1024**3:\n        raise RuntimeError(\'Less than 22 GiB temporary disk available for isolated dependencies and model\')\n    chosen = eligible[0]\n    context = 16384 if chosen[\'total_mib\'] < 22000 else 32768\n    # One prefill step for the whole prompt: on T4, chunked prefill (2.048) made an 8.4k-token\n    # synthesis prompt take ~230 s against the upstream 180 s client timeout.\n    settings = {\'gpu\': chosen, \'context\': context,\n                \'dtype\': \'half\', \'tensor_parallel_size\': 1, \'max_num_seqs\': 1,\n                \'vllm_engine\': \'V0\',\n                \'enable_chunked_prefill\': False, \'max_num_batched_tokens\': context,\n                \'max_output_tokens\': {\'self_state\': 512, \'consolidate\': 512, \'synthesizer\': 1024},\n                **HARNESS_LIMITS, \'client_retry_attempts\': 3,\n                \'model\': MODEL, \'model_revision\': REVISION, \'vllm\': VLLM_VERSION,\n                \'sampling\': {\'temperature\': 0.7, \'top_p\': 0.8, \'top_k\': 20, \'min_p\': 0.0, \'seed\': 20261026},\n                \'context_overflow\': \'fatal, never truncate\', \'concurrency\': 1}\n    dump(ROOT / \'settings.json\', settings)\n    return settings\n\n\ndef find_archive():\n    candidates = list(INPUT.rglob(\'rclagent-kaggle-bundle.zip\'))\n    if len(candidates) == 1:\n        return candidates[0]\n    if candidates:\n        raise RuntimeError(\'Multiple bundle archives attached\')\n    # Kaggle Dataset ingestion expands ZIPs. Bind the expanded content to the\n    # manifest digest embedded in the notebook, then verify every member.\n    manifests = [p for p in INPUT.rglob(\'manifest.json\') if (p.parent / \'upstream/coordinator.py\').is_file()]\n    if len(manifests) != 1:\n        raise RuntimeError(f\'Attach exactly one private bundle dataset; found {len(manifests)} expanded packages\')\n    return manifests[0].parent\n\n\ndef unpack(archive, expected_sha256, expected_manifest_sha256):\n    archive = Path(archive)\n    if archive.is_file() and digest(archive) != expected_sha256:\n        raise RuntimeError(\'Bundle SHA256 mismatch\')\n    package = ROOT / \'package\'\n    if package.exists():\n        # Safe reuse for bootstrap; execution is separately protected by run-once.json.\n        manifest = json.loads((package / \'manifest.json\').read_text())\n    else:\n        package.mkdir(parents=True)\n        if archive.is_dir():\n            if digest(archive / \'manifest.json\') != expected_manifest_sha256:\n                raise RuntimeError(\'Expanded Kaggle manifest SHA256 mismatch\')\n            manifest = json.loads((archive / \'manifest.json\').read_text())\n            for relative in [*manifest[\'files\'], \'manifest.json\']:\n                destination = (package / relative).resolve()\n                if not destination.is_relative_to(package.resolve()):\n                    raise RuntimeError(\'Unsafe manifest path\')\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copyfile(archive / relative, destination)\n        else:\n            with zipfile.ZipFile(archive) as zipped:\n                for entry in zipped.infolist():\n                    target = (package / entry.filename).resolve()\n                    if not target.is_relative_to(package.resolve()):\n                        raise RuntimeError(\'Unsafe zip path\')\n                zipped.extractall(package)\n        manifest = json.loads((package / \'manifest.json\').read_text())\n    if digest(package / \'manifest.json\') != expected_manifest_sha256:\n        raise RuntimeError(\'Manifest SHA256 mismatch\')\n    if manifest[\'upstream_commit\'] != COMMIT or manifest[\'model_revision\'] != REVISION:\n        raise RuntimeError(\'Code/model revision mismatch\')\n    actual = {p.relative_to(package).as_posix() for p in package.rglob(\'*\') if p.is_file()}\n    if actual != set(manifest[\'files\']) | {\'manifest.json\'}:\n        raise RuntimeError(\'Unexpected or missing package members\')\n    for relative, entry in manifest[\'files\'].items():\n        if digest(package / relative) != entry[\'sha256\']:\n            raise RuntimeError(\'Package member hash mismatch: \' + relative)\n    if manifest[\'files\'][\'kaggle-runtime.py\'][\'sha256\'] != digest(__file__):\n        raise RuntimeError(\'Notebook runtime differs from hashed package runtime\')\n    dump(ROOT / \'input-verification.json\', {\'source\': str(archive),\n        \'container\': \'expanded Kaggle dataset\' if archive.is_dir() else \'zip\',\n        \'manifest_sha256\': expected_manifest_sha256, \'all_member_hashes_verified\': True,\n        \'original_bundle_sha256\': expected_sha256,\n        \'container_zip_hash_checked\': archive.is_file()})\n    return package\n\n\ndef clean_env():\n    env = os.environ.copy()\n    for key in list(env):\n        if key.startswith(\'LLM_\') or key in (\'HF_TOKEN\', \'HUGGING_FACE_HUB_TOKEN\', \'HUGGINGFACEHUB_API_TOKEN\', \'VLLM_API_KEY\',\n                                           \'OPENAI_API_KEY\', \'ANTHROPIC_API_KEY\', \'DASHSCOPE_API_KEY\'):\n            env.pop(key)\n    env.update(HF_HUB_DISABLE_IMPLICIT_TOKEN=\'1\', HF_HUB_DISABLE_TELEMETRY=\'1\',\n               HF_HOME=str(TEMP / \'huggingface\'), PIP_DISABLE_PIP_VERSION_CHECK=\'1\',\n               DO_NOT_TRACK=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n               PYTHONDONTWRITEBYTECODE=\'1\',\n               # The author\'s code joins sets of service names into prompts; a fixed hash seed fixes their order.\n               PYTHONHASHSEED=\'0\')\n    return env\n\n\ndef install(expected_sha256, expected_manifest_sha256):\n    kaggle_only()\n    if not (ROOT / \'settings.json\').exists():\n        raise RuntimeError(\'Run environment cell A first\')\n    TEMP.mkdir(parents=True, exist_ok=True)\n    package = unpack(find_archive(), expected_sha256, expected_manifest_sha256)\n    if not PYTHON.exists():\n        # Kaggle\'s Debian Python may omit ensurepip. Create a bare venv, then\n        # use base pip\'s supported --python option to seed only that venv.\n        venv.EnvBuilder(with_pip=False).create(TEMP / \'env\')\n    marker = TEMP / \'pins.json\'\n    if marker.exists() and json.loads(marker.read_text()) != PINS:\n        raise RuntimeError(\'Existing isolated environment has different pins; stop without altering it\')\n    if not marker.exists():\n        with (ROOT / \'install.log\').open(\'w\', encoding=\'utf-8\') as log:\n            subprocess.run([sys.executable, \'-m\', \'pip\', \'--python\', str(PYTHON),\n                            \'install\', \'--index-url\', \'https://pypi.org/simple\',\n                            \'--no-cache-dir\', \'pip==25.2\'], env=clean_env(),\n                           stdout=log, stderr=subprocess.STDOUT, check=True, timeout=180)\n            subprocess.run([str(PYTHON), \'-m\', \'pip\', \'install\', \'--index-url\', \'https://pypi.org/simple\', \'--no-cache-dir\', *PINS],\n                           env=clean_env(), stdout=log, stderr=subprocess.STDOUT, check=True, timeout=1500)\n        dump(marker, PINS)\n    check = subprocess.run([str(PYTHON), \'-m\', \'pip\', \'check\'], capture_output=True, text=True, env=clean_env())\n    (ROOT / \'pip-check.txt\').write_text(check.stdout + check.stderr, encoding=\'utf-8\')\n    check.check_returncode()\n    freeze = subprocess.run([str(PYTHON), \'-m\', \'pip\', \'freeze\', \'--all\'], capture_output=True, text=True, check=True, env=clean_env())\n    (ROOT / \'pip-freeze.txt\').write_text(freeze.stdout, encoding=\'utf-8\')\n    probe = subprocess.run([str(PYTHON), \'-c\', \'import json,torch,transformers,vllm,pandas; print(json.dumps({"torch":torch.__version__,"cuda":torch.version.cuda,"vllm":vllm.__version__,"transformers":transformers.__version__,"pandas":pandas.__version__,"gpu_count":torch.cuda.device_count(),"cuda_available":torch.cuda.is_available()}))\'],\n                           capture_output=True, text=True, env=clean_env(), check=True, timeout=180)\n    (ROOT / \'environment-isolated.json\').write_text(probe.stdout, encoding=\'utf-8\')\n    if not json.loads(probe.stdout)[\'cuda_available\']:\n        raise RuntimeError(\'Isolated PyTorch cannot access CUDA; no model will be downloaded or run\')\n    settings = json.loads((ROOT / \'settings.json\').read_text())\n    kernel_env = clean_env()\n    kernel_env.update(CUDA_VISIBLE_DEVICES=str(settings[\'gpu\'][\'index\']), VLLM_USE_V1=\'0\',\n                      VLLM_NO_USAGE_STATS=\'1\', VLLM_DO_NOT_TRACK=\'1\')\n    evidence = ROOT / \'compatibility\'\n    checked = evidence / \'status.json\'\n    if not checked.exists():\n        subprocess.run([str(PYTHON), str(package / \'verify-t4-prefix-kernel.py\'),\n                        \'--evidence\', str(evidence)], env=kernel_env, check=True, timeout=600)\n    if json.loads(checked.read_text())[\'status\'] != \'passed\':\n        raise RuntimeError(\'GPU prefix compatibility verification failed\')\n    print(probe.stdout)\n    print(\'Installed in /tmp; Kaggle base packages were not modified.\')\n    return package\n\n\ndef assert_free_port(port):\n    with socket.socket() as probe:\n        probe.bind((\'127.0.0.1\', port))\n\n\nclass Processes:\n    def __init__(self, result):\n        self.result = result\n        self.children = []\n        self.files = []\n\n    def start(self, name, argv, env):\n        log = (self.result / (name + \'.log\')).open(\'wb\')\n        self.files.append(log)\n        child = subprocess.Popen(argv, env=env, cwd=self.result,\n                                 stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        self.children.append({\'name\': name, \'process\': child, \'argv\': argv})\n        self.record()\n        return child\n\n    def record(self):\n        dump(self.result / \'processes.json\', [{\'name\': item[\'name\'], \'pid\': item[\'process\'].pid,\n            \'port\': 8000 if item[\'name\'] == \'vllm\' else 5000 if item[\'name\'].startswith(\'tools-\') else None,\n            \'cwd\': str(self.result), \'argv\': item[\'argv\'], \'exit\': item[\'process\'].poll()} for item in self.children])\n\n    def stop(self, child):\n        # Only process groups created by start_new_session=True in this run.\n        try:\n            os.killpg(child.pid, signal.SIGTERM)\n        except ProcessLookupError:\n            pass\n        try:\n            child.wait(timeout=12)\n        except subprocess.TimeoutExpired:\n            pass\n        # The leader can exit before its descendants. Check the owned group too.\n        try:\n            os.killpg(child.pid, 0)\n        except ProcessLookupError:\n            pass\n        else:\n            os.killpg(child.pid, signal.SIGKILL)\n        try:\n            child.wait(timeout=5)\n        except subprocess.TimeoutExpired:\n            pass\n        self.record()\n\n    def cleanup(self):\n        for item in reversed(self.children):\n            self.stop(item[\'process\'])\n        for handle in self.files:\n            handle.close()\n\n\ndef wait_ready(child, path, timeout):\n    import urllib.request\n    end = time.monotonic() + timeout\n    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))\n    while time.monotonic() < end:\n        if child.poll() is not None:\n            raise RuntimeError(\'Owned server exited before ready; inspect its log\')\n        try:\n            with opener.open(path, timeout=2) as response:\n                if response.status == 200:\n                    return\n        except (OSError, ValueError):\n            time.sleep(1)\n    raise TimeoutError(\'Server startup timed out\')\n\n\ndef assert_unserved(port):\n    # A connect probe, not a bind: the last server\'s connections may still sit in TIME_WAIT on this port, which\n    # blocks a plain bind on Linux although nothing answers there any more.\n    with socket.socket() as probe:\n        probe.settimeout(2)\n        if probe.connect_ex((\'127.0.0.1\', port)) == 0:\n            raise RuntimeError(f\'Port {port} still held by another process; refusing to start a tool server\')\n\n\ndef start_tool_server(proc, name, argv, env, probe_url, timeout):\n    """Start a tool server on port 5000; one that never becomes ready is stopped before the error propagates."""\n    assert_unserved(5000)\n    child = proc.start(name, argv, env)\n    try:\n        wait_ready(child, probe_url, timeout)\n    except BaseException:\n        proc.stop(child)\n        raise\n    return child\n\n\ndef score_runs(package, result, progress, python, env):\n    """Score each passed run with the author\'s evaluate.py on its own; return the run_ids whose scoring failed."""\n    shutil.rmtree(result / \'scoring\', ignore_errors=True)\n    failed = []\n    for entry in run_entries(package):\n        if progress[\'traces\'][entry[\'run_id\']][\'status\'] != \'passed\':\n            continue\n        trace_dir, scoring = package / entry[\'dir\'], result / \'scoring\' / entry[\'run_id\']\n        exit_code, error = None, None\n        try:\n            (scoring / \'result\').mkdir(parents=True)\n            shutil.copyfile(trace_dir / \'scoring/groundtruth.csv\', scoring / \'groundtruth.csv\')\n            shutil.copyfile(trace_dir / \'input/error_traces.txt\', scoring / \'error_traces.txt\')\n            shutil.copyfile(result / \'traces\' / entry[\'run_id\'] / \'result.json\', scoring / \'result/conversation_trace_1.txt\')\n            with (scoring / \'author-score.txt\').open(\'w\', encoding=\'utf-8\') as log:\n                exit_code = subprocess.run([str(python), str(package / \'upstream/evaluate.py\'), str(scoring), \'result\'],\n                                           env=env, stdout=log, stderr=subprocess.STDOUT, timeout=60).returncode\n        except Exception as exc:\n            error = repr(exc)\n        dump(scoring / \'scorer-status.json\', {\'exit\': exit_code, \'error\': error})\n        if exit_code or error:\n            failed.append(entry[\'run_id\'])\n    return failed\n\n\nclass Resources:\n    def __init__(self, processes, result):\n        self.processes, self.result = processes, result\n        self.done = threading.Event()\n        self.phase = \'setup\'\n        self.peaks = {}\n        self.thread = threading.Thread(target=self.sample, daemon=True)\n\n    def sample(self):\n        # This stdlib thread uses /proc, so no package changes to the notebook kernel.\n        while not self.done.is_set():\n            try:\n                self.sample_once()\n            except Exception as exc:\n                with (self.result / \'monitor-errors.log\').open(\'a\', encoding=\'utf-8\') as f:\n                    f.write(repr(exc) + \'\\n\')\n            self.done.wait(1)\n\n    def sample_once(self):\n        roots = {item[\'process\'].pid for item in self.processes.children}\n        rows = {}\n        for procdir in Path(\'/proc\').glob(\'[0-9]*\'):\n            try:\n                stat = (procdir / \'stat\').read_text()\n                tail = stat[stat.rfind(\')\')+2:].split()\n                rows[int(procdir.name)] = (int(tail[1]), int(tail[21]) * os.sysconf(\'SC_PAGE_SIZE\'))\n            except (OSError, ValueError, IndexError):\n                continue\n        owned = set(roots)\n        for _ in range(20):\n            more = {pid for pid, (ppid, _) in rows.items() if ppid in owned}\n            if more <= owned:\n                break\n            owned |= more\n        ram = sum(rss for pid, (_, rss) in rows.items() if pid in owned)\n        gpu = {}\n        probe = subprocess.run([\'nvidia-smi\', \'--query-compute-apps=pid,used_gpu_memory,gpu_uuid\',\n                                \'--format=csv,noheader,nounits\'], capture_output=True, text=True, timeout=5)\n        if probe.returncode == 0:\n            for line in probe.stdout.splitlines():\n                parts = [x.strip() for x in line.split(\',\')]\n                if len(parts) == 3 and parts[0].isdigit() and int(parts[0]) in owned:\n                    try:\n                        gpu[parts[2]] = gpu.get(parts[2], 0) + float(parts[1])\n                    except ValueError:\n                        continue\n        peak = self.peaks.setdefault(self.phase, {\'sampled_peak_owned_rss_bytes\': 0, \'sampled_peak_owned_gpu_mib\': {}, \'samples\': 0})\n        peak[\'samples\'] += 1\n        peak[\'sampled_peak_owned_rss_bytes\'] = max(peak[\'sampled_peak_owned_rss_bytes\'], ram)\n        for uuid, mib in gpu.items():\n            peak[\'sampled_peak_owned_gpu_mib\'][uuid] = max(peak[\'sampled_peak_owned_gpu_mib\'].get(uuid, 0), mib)\n        with (self.result / \'resources.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\n            f.write(json.dumps({\'time\': time.time(), \'phase\': self.phase, \'owned_rss_bytes\': ram,\n                                \'owned_gpu_mib\': gpu or None, \'gpu_probe_exit\': probe.returncode}) + \'\\n\')\n\n    def finish(self):\n        self.done.set()\n        self.thread.join(timeout=7)\n        dump(self.result / \'resource-peaks.json\', {\'by_phase\': self.peaks,\n             \'method\': \'Owned process-tree RSS and nvidia-smi compute-app memory, sampled every ~1s. Device peaks remain separate; RSS may double-count shared pages. Missing GPU metrics are unknown, not zero. Brief allocation peaks between samples can be missed.\'})\n\n\ndef synthetic_data(path):\n    path.mkdir(parents=True, exist_ok=True)\n    for part in (\'trace/all\', \'metric/all\', \'log/all\'):\n        (path / part).mkdir(parents=True, exist_ok=True)\n    (path / \'trace/all/trace_jaeger-span.csv\').write_text(\'timestamp,cmdb_id,span_id,trace_id,duration,type,status_code,operation_name,parent_span\\n1700000000000,syntheticservice,synthetic-span,synthetic-trace,1,rpc,0,synthetic-operation,\\n\')\n    (path / \'metric/all/metrics.csv\').write_text(\'timestamp,node_id,service_name,kpi_name,value\\n1700000000,,syntheticservice,cpu,1.0\\n\')\n    (path / \'log/all/logs.csv\').write_text(\'log_id,timestamp,cmdb_id,log_name,value\\nsynthetic-log,1700000000,syntheticservice,synthetic,"severity: ERROR, message: SYNTHETIC_OK"\\n\')\n    for name in (\'node_service_map.pkl\', \'service_node_map.pkl\'):\n        (path / \'metric\' / name).write_bytes(b\'\\x80\\x04}\\x94.\')\n\n\ndef export_results(result):\n    archive = ROOT / \'rclagent-smoke-results.zip\'\n    allowed_extra = (\'environment-before.json\', \'settings.json\', \'environment-isolated.json\',\n                     \'pip-freeze.txt\', \'pip-check.txt\', \'install.log\', \'input-verification.json\')\n    with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as zipped:\n        for path in sorted(result.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, \'run/\' + path.relative_to(result).as_posix())\n        for name in allowed_extra:\n            if (ROOT / name).exists():\n                zipped.write(ROOT / name, name)\n        for path in sorted((ROOT / \'compatibility\').rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, \'compatibility/\' + path.relative_to(ROOT / \'compatibility\').as_posix())\n    (ROOT / \'rclagent-smoke-results.zip.sha256\').write_text(digest(archive) + \'  \' + archive.name + \'\\n\')\n    print(\'Results:\', archive, \'SHA256:\', digest(archive))\n    return archive\n\n\ndef run_entries(package):\n    """The manifest\'s runs (a trace under one seed and source condition), or one author-condition run per trace."""\n    manifest = json.loads((package / \'manifest.json\').read_text())\n    return manifest.get(\'runs\') or [dict(t, run_id=t[\'trace_id\']) for t in manifest[\'traces\']]\n\n\ndef harness_limits(manifest):\n    """The bundle\'s trace limits: the defaults, overridden by known positive integers in harness_limits."""\n    given = manifest.get(\'harness_limits\', {})\n    unknown = sorted(set(given) - set(HARNESS_LIMITS))\n    if unknown:\n        raise ValueError(\'Unknown harness limits: \' + \', \'.join(unknown))\n    for key, value in given.items():\n        if type(value) is not int or value < 1:\n            raise ValueError(f\'Harness limit {key} must be a positive integer, not {value!r}\')\n    return dict(HARNESS_LIMITS, **given)\n\n\ndef limit_options(limits):\n    return [part for key, flag in LIMIT_FLAGS.items() for part in (flag, str(limits[key]))]\n\n\ndef worker_options(entry):\n    """Worker flags of one run; a run without them keeps the author\'s tools and the frozen seed."""\n    options = []\n    for flag in (\'seed\', \'logs\', \'metric\'):\n        if flag in entry:\n            options += [\'--\' + flag, str(entry[flag])]\n    return options\n\n\ndef set_aside(path):\n    """Keep a partial directory from an interrupted attempt as evidence instead of overwriting it."""\n    if path.exists():\n        index = 0\n        while path.with_name(f\'{path.name}.previous-{index}\').exists():\n            index += 1\n        path.rename(path.with_name(f\'{path.name}.previous-{index}\'))\n\n\ndef load_progress(path, bundle_sha256):\n    if not path.exists():\n        return {\'bundle_sha256\': bundle_sha256, \'traces\': {}}\n    progress = json.loads(path.read_text())\n    if progress[\'bundle_sha256\'] != bundle_sha256:\n        raise RuntimeError(\'Recorded progress belongs to another bundle; refusing to mix runs\')\n    return progress\n\n\ndef run_traces(package, result, bundle_sha256, run_trace, model_alive):\n    """Run every packaged run once, in package order, recording progress after each state change.\n\n    A run is a trace under one seed and source condition; progress, outputs and scores are keyed by its run_id\n    (the trace_id when the manifest lists no runs).\n\n    A trace with a final status (passed or failed) is skipped when the batch is resumed; one cut off mid-run\n    (still running) runs again, its partial output set aside. A failed trace is recorded with its reason and the\n    batch moves on, without retry or configuration change. A dead model server stops the batch with the\n    remaining traces pending; a trace that fails while the server process has died is left running, so a resume\n    reruns it instead of booking an infrastructure failure as the model\'s. A second server death in the same trace\n    books it failed, so a prompt that crashes the server cannot block the batch on every resume.\n    """\n    path = result / \'progress.json\'\n    progress = load_progress(path, bundle_sha256)\n    for entry in run_entries(package):\n        state = progress[\'traces\'].setdefault(entry[\'run_id\'], {\'case\': entry[\'case\'], \'attempts\': []})\n        if state.get(\'status\') in (\'passed\', \'failed\'):\n            continue\n        if not model_alive():\n            dump(path, progress)\n            raise RuntimeError(\'Model server is not running; remaining traces stay pending for a resumed run\')\n        out = result / \'traces\' / entry[\'run_id\']\n        if state.get(\'status\') == \'running\':\n            state[\'attempts\'][-1][\'outcome\'] = \'interrupted\'\n            set_aside(out)\n        state[\'status\'] = \'running\'\n        state[\'attempts\'].append({\'started\': time.time()})\n        dump(path, progress)\n        try:\n            run_trace(package / entry[\'dir\'], out, entry)\n            state[\'status\'] = \'passed\'\n        except Exception as exc:  # An interrupt (BaseException) leaves the trace running, so a resume reruns it.\n            if not model_alive() and not any(a.get(\'server_died\') for a in state[\'attempts\'][:-1]):\n                state[\'attempts\'][-1].update(outcome=\'interrupted\', server_died=True, reason=repr(exc),\n                                             finished=time.time())\n                dump(path, progress)\n                raise RuntimeError(\'Model server died during a trace; it and the remaining traces run again on resume\')\n            state.update(status=\'failed\', reason=repr(exc))\n        state[\'attempts\'][-1].update(outcome=state[\'status\'], finished=time.time())\n        dump(path, progress)\n    return progress\n\n\ndef import_prior(result, bundle_sha256):\n    """Continue a batch from the results of an earlier session attached as input, zip or expanded."""\n    sources = [p for p in INPUT.rglob(\'rclagent-smoke-results.zip\')] + \\\n              [p.parent.parent for p in INPUT.rglob(\'progress.json\') if p.parent.name == \'run\']\n    if not sources:\n        return False\n    if len(sources) != 1:\n        raise RuntimeError(f\'Attach at most one earlier result set; found {len(sources)}\')\n    if (result / \'progress.json\').exists():\n        raise RuntimeError(\'This session already has progress; refusing to merge an attached result set\')\n    source = sources[0]\n    zipped = zipfile.ZipFile(source) if source.is_file() else None\n    try:\n        progress = json.loads(zipped.read(\'run/progress.json\') if zipped else (source / \'run/progress.json\').read_text())\n        if progress[\'bundle_sha256\'] != bundle_sha256:\n            raise RuntimeError(\'Attached results belong to another bundle\')\n        if zipped:\n            for entry in zipped.infolist():\n                if entry.filename.startswith(\'run/traces/\') and not entry.is_dir():\n                    target = (result / entry.filename.removeprefix(\'run/\')).resolve()\n                    if not target.is_relative_to(result.resolve()):\n                        raise RuntimeError(\'Unsafe path in attached results\')\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    target.write_bytes(zipped.read(entry))\n        elif (source / \'run/traces\').is_dir():\n            shutil.copytree(source / \'run/traces\', result / \'traces\', dirs_exist_ok=True)\n    finally:\n        if zipped:\n            zipped.close()\n    dump(result / \'progress.json\', progress)\n    return True\n\n\ndef preparation_failure(stage):\n    """Make A/B/early-C errors downloadable even before a model process exists."""\n    output = ROOT / (\'preparation-\' + stage)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \'error.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n    dump(output / \'status.json\', {\'status\': \'failed\', \'stage\': stage, \'model_run\': \'not-started\'})\n    return export_results(output)\n\n\ndef run_once(expected_sha256, expected_manifest_sha256):\n    kaggle_only()\n    package = unpack(find_archive(), expected_sha256, expected_manifest_sha256)\n    settings = json.loads((ROOT / \'settings.json\').read_text())\n    limits = harness_limits(json.loads((package / \'manifest.json\').read_text()))\n    settings.update(limits)  # Record the limits this bundle\'s traces run under.\n    dump(ROOT / \'settings.json\', settings)\n    if json.loads((ROOT / \'compatibility/status.json\').read_text())[\'status\'] != \'passed\':\n        raise RuntimeError(\'Run the synthetic GPU kernel compatibility check before starting the model\')\n    marker = ROOT / \'run-once.json\'\n    if marker.exists():\n        state = json.loads(marker.read_text())\n        if state[\'bundle_sha256\'] != expected_sha256:\n            raise RuntimeError(\'A run of another bundle exists here; stop without altering it\')\n        state[\'resumed\'] = state.get(\'resumed\', []) + [time.time()]\n    else:\n        state = {\'started\': time.time(), \'bundle_sha256\': expected_sha256,\n                 \'policy\': \'One pass over the packaged traces. A resumed run continues pending traces only; a \'\n                           \'finished trace, passed or failed, never runs again. A new pass needs a new user decision.\'}\n    dump(marker, state)\n    result = ROOT / \'run\'\n    result.mkdir(exist_ok=True)\n    imported = import_prior(result, expected_sha256)\n    proc = Processes(result)\n    monitor = Resources(proc, result)\n    atexit.register(proc.cleanup)\n    status = {\'status\': \'failed\', \'stage\': \'C\', \'started\': time.time(), \'score\': \'not-run\',\n              \'resumed_from_attached_results\': imported}\n    env = clean_env()\n    env.update(CUDA_VISIBLE_DEVICES=str(settings[\'gpu\'][\'index\']), LLM_MODEL=MODEL,\n               LLM_API_URL=\'http://127.0.0.1:8000/v1/chat/completions\',\n               LLM_API_KEY=\'local-placeholder\', MAX_AGENT_PARALLEL=\'1\', DATASET_TYPE=\'re2ob\',\n               TOOL_SERVER_HOST=\'127.0.0.1\', TOOL_SERVER_PORT=\'5000\', LLM_FORCE_STREAM=\'false\',\n               VLLM_USE_V1=\'0\', VLLM_NO_USAGE_STATS=\'1\', VLLM_DO_NOT_TRACK=\'1\')\n    monitor.thread.start()\n    progress = None\n    try:\n        for port in (8000, 5000):\n            assert_free_port(port)  # Never kill another session or increment ports.\n        # Downloads weights/config only from the public, non-gated official repo.\n        download = \'from huggingface_hub import snapshot_download; print(snapshot_download(repo_id=\' + repr(MODEL) + \',revision=\' + repr(REVISION) + \',token=False))\'\n        with (result / \'model-download.log\').open(\'w\', encoding=\'utf-8\') as log:\n            subprocess.run([str(PYTHON), \'-c\', download], env=env, stdout=log,\n                           stderr=subprocess.STDOUT, check=True, timeout=1200)\n        snapshot = TEMP / \'huggingface/hub\' / (\'models--\' + MODEL.replace(\'/\', \'--\')) / \'snapshots\' / REVISION\n        if not snapshot.is_dir():\n            raise RuntimeError(\'Pinned model snapshot not found\')\n        dump(result / \'model-files.json\', {p.name: {\'bytes\': p.stat().st_size, \'sha256\': digest(p)} for p in snapshot.iterdir() if p.is_file()})\n        env.update(HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\')\n        server = proc.start(\'vllm\', [str(PYTHON), \'-m\', \'vllm.entrypoints.openai.api_server\',\n            \'--model\', str(snapshot), \'--tokenizer\', str(snapshot), \'--served-model-name\', MODEL,\n            \'--revision\', REVISION, \'--tokenizer-revision\', REVISION,\n            \'--host\', \'127.0.0.1\', \'--port\', \'8000\', \'--dtype\', \'half\',\n            \'--tensor-parallel-size\', \'1\', \'--max-num-seqs\', \'1\',\n            \'--max-model-len\', str(settings[\'context\']), \'--gpu-memory-utilization\', \'0.85\',\n            \'--max-num-batched-tokens\', str(settings[\'max_num_batched_tokens\']), \'--enforce-eager\',\n            \'--enable-auto-tool-choice\', \'--tool-call-parser\', \'hermes\',\n            \'--generation-config\', \'vllm\', \'--disable-log-requests\'], env)\n        wait_ready(server, \'http://127.0.0.1:8000/health\', 600)\n        if \'Chunked prefill is enabled\' in (result / \'vllm.log\').read_text(encoding=\'utf-8\', errors=\'replace\'):\n            raise RuntimeError(\'vLLM turned chunked prefill on; each prompt must be prefilled in one step\')\n        print(\'C: pinned local model server ready; single GPU, FP16, context\', settings[\'context\'])\n        status[\'stage\'] = \'D\'\n        monitor.phase = \'protocol\'\n        synth = TEMP / \'synthetic\'\n        synthetic_data(synth)\n        def start_tools(name, data):\n            return start_tool_server(proc, name, [str(PYTHON), str(package / \'upstream/tool_server.py\')],\n                                     dict(env, DATA_ROOT=str(data)), \'http://127.0.0.1:5000/search_span?span_id=probe\', 90)\n        tools = start_tools(\'tools-synthetic\', synth)\n        def worker(phase, data, out, timeout, trace_dir=None, options=()):\n            out.mkdir(parents=True)\n            worker_env = dict(env, DATA_ROOT=str(data))\n            child = proc.start(\'worker-\' + out.name, [str(PYTHON), str(package / \'smoke-worker.py\'), phase,\n                \'--package\', str(package), \'--output\', str(out), \'--model-path\', str(snapshot),\n                \'--context\', str(settings[\'context\'])] + ([\'--trace-dir\', str(trace_dir)] if trace_dir else []) +\n                list(options), worker_env)\n            try:\n                exit_code = child.wait(timeout=timeout)\n            except subprocess.TimeoutExpired:\n                proc.stop(child)\n                raise TimeoutError(phase + \' wall-clock limit reached; partial logs retained\')\n            proc.record()\n            state = json.loads((out / \'worker-status.json\').read_text()) if (out / \'worker-status.json\').exists() else {}\n            if exit_code or state.get(\'status\') != \'passed\':\n                raise RuntimeError(f"{phase} failed: {state.get(\'fatal\') or state.get(\'error\') or \'exit \' + str(exit_code)}")\n        protocol = result / \'protocol\'\n        set_aside(protocol)  # A resumed session checks its own server again.\n        worker(\'protocol\', synth, protocol, 480)\n        proc.stop(tools)\n        print(\'D: synthetic structured tool calls, tool-role round trip, print_results and request field checks passed\')\n        status[\'stage\'] = \'E\'\n        monitor.phase = \'trace\'\n        def run_trace(trace_dir, out, entry):\n            started = time.monotonic()  # Per-trace deadline, tool server start included.\n            tools = start_tools(\'tools-\' + out.name, trace_dir / \'input\')\n            try:\n                worker(\'trace\', trace_dir / \'input\', out,\n                       max(0.1, limits[\'rcl_deadline_seconds\']-(time.monotonic()-started)), trace_dir,\n                       worker_options(entry) + limit_options(limits))\n            finally:\n                proc.stop(tools)\n            print(\'E:\', out.name, \'completed in\', round(time.monotonic()-started), \'s\')\n        progress = run_traces(package, result, expected_sha256, run_trace, lambda: server.poll() is None)\n        print(\'E: every packaged trace has a final status; failed traces are recorded, not retried\')\n        status[\'stage\'] = \'F\'\n        monitor.phase = \'scoring\'\n        # Stop model inference before opening any groundtruth file.\n        proc.stop(server)\n        failed = score_runs(package, result, progress, PYTHON, env)\n        if failed:\n            raise RuntimeError(\'Author scorer failed for: \' + \', \'.join(failed))\n        status.update(status=\'passed\', score=\'author scorer per passed trace; descriptive pilot result, not an accuracy estimate\')\n    except BaseException as exc:\n        status[\'error\'] = repr(exc)\n        (result / \'error.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        print(\'STOP:\', status[\'stage\'], repr(exc))\n    finally:\n        try:\n            proc.cleanup()\n        finally:\n            monitor.finish()\n            status[\'finished\'] = time.time()\n            status[\'elapsed_seconds\'] = status[\'finished\']-status[\'started\']\n            if (result / \'progress.json\').exists():\n                progress = json.loads((result / \'progress.json\').read_text())\n                outcomes = [t.get(\'status\', \'pending\') for t in progress[\'traces\'].values()]\n                status[\'traces\'] = {s: outcomes.count(s) for s in sorted(set(outcomes))}\n            status[\'observed_requests\'] = {}\n            for events in sorted(result.glob(\'*/events.jsonl\')) + sorted(result.glob(\'traces/*/events.jsonl\')):\n                count = 0\n                for line in events.read_text(encoding=\'utf-8\').splitlines():\n                    try:\n                        count += json.loads(line).get(\'kind\') == \'request\'\n                    except json.JSONDecodeError:\n                        continue\n                status[\'observed_requests\'][events.parent.relative_to(result).as_posix()] = count\n            dump(result / \'status.json\', status)\n            archive = export_results(result)\n            atexit.unregister(proc.cleanup)\n    if status[\'status\'] != \'passed\':\n        raise RuntimeError(\'Run stopped; download partial results, then resume with them attached: \' + str(archive))\n    return archive\n'
assert hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest() == 'e04281a34faa4d225d4e9799e425f2013dd6e052ff6cb90e9155e1c4787f3936'
runtime_path = Path('/tmp/rclagent-kaggle-runtime.py')
runtime_path.write_bytes(RUNTIME_SOURCE.encode())
spec = importlib.util.spec_from_file_location('rclagent_kaggle_runtime', runtime_path)
rcl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rcl)
print('Expected bundle SHA256:', EXPECTED_BUNDLE_SHA256)


## A. Kiểm môi trường thật

Ghi riêng từng GPU, VRAM, RAM, đĩa, Python/PyTorch/CUDA và driver. Chọn thiết bị
đầu tiên đáp ứng yêu cầu; không cộng VRAM. Context: 16.384 token khi VRAM <22.000 MiB,
32.768 token khi VRAM ≥22.000 MiB. Mọi cấu hình được lưu trong settings.json.


In [ ]:
try:
    settings = rcl.environment()
except BaseException:
    rcl.preparation_failure('A')
    raise


## B. Kiểm hash, cài môi trường cô lập và ghi phiên bản

Ghim vLLM 0.10.2, Transformers 4.55.4 và các thư viện trực tiếp; pip ghi phiên bản
phụ thuộc thực tế. Venv và model cache ở /tmp, không thay thư viện Kaggle sẵn có.
Model duy nhất là `Qwen/Qwen3-4B-Instruct-2507`, revision
`cdbee75f17c01a7cc42f958dc650907174af0554` từ tài khoản Qwen chính thức.
Không cần API key hoặc HF token. Cài đặt có thể mất vài phút; xem install.log nếu lỗi.

Sau cài đặt, chạy kiểm kernel tổng hợp trên GPU (không nạp model/telemetry):
bản gốc phải tái hiện lỗi shared memory trên T4, bản vá phải khớp dense causal
attention FP32 (atol/rtol 1e-3) cho 15 trường hợp. Lưu source gốc, unified diff,
hash và kết quả số học trong compatibility/. Sai source/hash hoặc kiểm thất bại thì dừng.


In [ ]:
try:
    package = rcl.install(EXPECTED_BUNDLE_SHA256, EXPECTED_MANIFEST_SHA256)
except BaseException:
    rcl.preparation_failure('B')
    raise


## C–F. Server localhost → gate tổng hợp → từng trace → xuất kết quả và dọn process

Các bước C–F nằm trong cùng một cell có `try/finally` để dọn process cả khi lỗi hoặc
người dùng Interrupt. Không có server nào được giữ lại giữa các cell.

**C:** tải một revision, nạp FP16 trên một GPU; vLLM V0 + parser Hermes chính thức,
chỉ lắng nghe localhost. Không paid/cloud API, tunnel hay backbone thứ hai.

**D:** dùng telemetry tổng hợp: structured tool_calls qua client gốc, thực thi
search_logs, trả kết quả bằng role=tool, gọi print_results theo schema ≥10 mục.
Chuyển enable_thinking sang chat_template_kwargs; chặn trường lạ. D thất bại thì
không khởi động tool server trên dữ liệu benchmark; không có parser văn bản vá lỗi.

**E:** lần lượt từng trace trong `manifest.json` của gói. Mỗi trace có tool server riêng
trên dữ liệu của chính nó, trần 320 model requests **tính cả retry** và 40 phút riêng,
concurrency=1, tối đa 8 vòng tool/span; retry client gốc tối đa 3 attempt.
Output tối đa 512 token cho self-state/consolidation, 1.024 cho synthesizer.
Input+output vượt context, output bị cắt, OOM, sai schema hoặc không có kết quả hợp lệ
đều dừng. Query tool cho mọi service được trả như dữ liệu đầy đủ; timestamp ngoài các giây
của trace dừng vì gói không bao phủ. Kết quả search_logs trên 2.048 token được rút gọn
theo ngân sách ở trên; bản gốc và bản model nhận đều được ghi trong events.jsonl.
Không cắt context, không tự tăng giới hạn, không chạy lại để chọn đáp án.
Một trace dừng (guard, deadline, lỗi HTTP) được ghi `failed` kèm lý do trong
`run/progress.json`, rồi lượt chuyển sang trace kế tiếp, không retry. Nếu model server
chết thì cả lượt dừng; các trace còn lại giữ trạng thái chờ để chạy tiếp.

**F:** lưu request/response, lỗi, số lời gọi, token khi server trả usage, thời gian,
RAM/VRAM đỉnh lấy mẫu (~1 giây). Chỉ sau khi E kết thúc và đã tắt model server mới
mở nhãn của từng trace đã qua rồi chạy scorer tác giả cho trace đó. File ZIP được xuất cả khi thất bại.
SIGKILL/khởi động lại kernel không cho Python thực hiện finally; lỗi và Interrupt thông thường có cleanup.


In [ ]:
from IPython.display import display, FileLink
try:
    result_zip = rcl.run_once(EXPECTED_BUNDLE_SHA256, EXPECTED_MANIFEST_SHA256)
except BaseException:
    if not (rcl.ROOT / 'rclagent-smoke-results.zip').exists():
        rcl.preparation_failure('C')
    raise
finally:
    archive = rcl.ROOT / 'rclagent-smoke-results.zip'
    if archive.exists():
        display(FileLink('rclagent-smoke/rclagent-smoke-results.zip'))
        print('Tải file ZIP này trong Output/Files; chứa cả phần đã hoàn thành khi lỗi.')


## Đọc kết quả

Tải `/kaggle/working/rclagent-smoke/rclagent-smoke-results.zip` và file .sha256.
Trong ZIP: `run/status.json`, `run/progress.json` (trạng thái và lý do của từng trace),
`run/traces/<trace_id>/worker-status.json`, `events.jsonl`, `result.json` (nếu có),
`run/scoring/<trace_id>/author-score.txt` (chỉ trace đã qua), resource-peaks.json,
resources.jsonl và log server/process. Marker protocol_pass nằm trong
`run/protocol/events.jsonl`. Thất bại luôn được giữ, không loại khỏi báo cáo.

Log tool có thể rỗng theo cách lọc upstream; không bổ sung tín hiệu. Mỗi trace được chấm
riêng, không dùng best-of-traces. Đầu ra cuối không cung cấp confidence cho H1. Kết quả đúng/sai
của backbone 4B này không tự chứng minh chất lượng phương pháp gốc.

Chuẩn bị CPU đã đối chiếu tool với dữ liệu đầy đủ; xem cpu-equivalence.json trong
bundle. Kết quả native mới nhất, nếu có, được ghi ở đầu notebook. Các mục chưa qua
gate D/E vẫn chưa được xác minh; không suy pass từ fixture hoặc server health.

Nguồn: [RCLAgent upstream](https://github.com/LLM4AIOps/RCLAgent-V2),
[RCAEval benchmark](https://zenodo.org/records/14590730),
[Qwen model](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507),
[vLLM 0.10.2 tool calling](https://docs.vllm.ai/en/v0.10.2/features/tool_calling.html).
